<h1>Chapter 5 - Tools</h1>
<i>Giving an Agent access to the Environment through Tool Usage</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 5 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma3:12b &

# Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use:

In [4]:
import os
from illustrated_agents.llm import LLM

# Ollama
llm = LLM(model="ollama/gemma3:12b")

# Llama.cpp server
# llm = LLM(model="openai/gemma-3-12b-it-Q4_K_M", api_base="http://localhost:8080", api_key="sk-no-key-required")

# Llama-cpp-python server
# llm = LLM(model="openai/gemma-3-12b-it-Q4_K_M.gguf", api_base="http://localhost:8000/v1/", api_key="sk-no-key-required")

# LM Studio
# llm = LLM(model="lm_studio/gemma-3-12b-it", api_base="http://localhost:1234/v1", api_key="sk-no-key-required")

# Google's Gemini / Gemma
# os.environ['GEMINI_API_KEY'] = "YOUR_GEMINI_API_KEY"
# llm = LLM(model="gemini/gemini-2.5-flash")
# llm = LLM(model="gemini/gemma-3-12b-it")

# Adding **`Tools`**

In the previous chapter, we added the `Memory` module to your `TinyAgent`. In this chapter, we will cover how to give it access to tools:

![../images/ch5.png](../images/ch5.png)

Tool usage, as covered in the book, has a wide range of methodologies to choose from (function calling, JSON, MCP, skills, etc.). Implementing them can therefore be a bit tricky, especially when you have are using an LLM that was not trained specifically for one of those techniques. Fortunately, LLMs are quite capable these days and even smaller models can follow instructions quite accurately. This gives us a bit of freedom with prompting techniques.

Throughout this notebook we are going with the following type of workflow:

* Tool Creation
* Tool Definition
* Tool Selection
* Tool Calling
* Tool Output Processing

![../images/ch5_tools.png](../images/ch5_tools.png)



# Tool Creation

Before we create a tool registry for your `TinyAgent`, let's first explore how we can create a tool ourselves. In its most basic form, a tool is nothing more than a function which may take in some arguments and outputs a string. In the following code block, we create a basic calculator that only adds value and a `get_weather` function that always give back the same weather for all locations.

In [5]:
def calculator(a: str, b: str) -> float:
    return float(a) + float(b)

def get_weather(location: str) -> str:
    return f"Weather in {location}: Sunny, 72°F"

Each function is a `Tool` and as shown, can be as simple (or complex) as you can think off. More complexity, however, might result in your `TinyAgent` having difficulties understanding how it works. What's nice about the above examples is that the `calculator` is a bit misleading, perhaps `add` or something similar would have been prefered. As such, you also want to give your tool some form of description that can be used by your `TinyAgent`. 

# Tool Definition

Defining your tool can be done in various ways. Using the docstrings is typically a nice option. For our purpose, we are going to keep it simple and create a very short description for each of the tools. To do so, let's also build the `Tools` class that manages how tools are defined and used.

We start small and give it the following functions:

* `add_tool` -- Add a tool for the Agent to use
* `prompt` -- A prompt that instructs the Agent on the tools that are available and how to call them
* `descriptions` -- The descriptions of each prompt

In [6]:
from typing import Callable


class Tools:
    """Tool registry for the Agent."""

    def __init__(self):
        self.tools = {}

    def add_tool(self, name: str, func: Callable, description: str):
        """Register a tool that the Agent can use.

        Arguments:
            name: The name of the tool.
            func: The function implementing the tool.
            description: A description of the tool.
        """
        self.tools[name] = {"function": func, "description": description}

    @property
    def descriptions(self):
        """Get descriptions of all registered tools."""
        return "\n".join(
            f"`{tool}`: {self.tools[tool]['description']}" for tool in self.tools
        )


When can then add the Tools, including their descriptions as follows:

In [7]:
# Register tools
tools = Tools()
tools.add_tool("calculator", calculator, "Adds two numbers: calculator(a, b)")
tools.add_tool("get_weather", get_weather, "Gets weather: get_weather(city)")

Note that because these are simple functions, we do not need much for the LLM to understand. When the tools grow in complexity and number of parameters, however, a more extensive description is needed that details the type of parameters, what they do, and what kind of output is to be expected.

We can view the descriptions of our tools:

In [8]:
print(tools.descriptions)

`calculator`: Adds two numbers: calculator(a, b)
`get_weather`: Gets weather: get_weather(city)


As covered in the book, responsing with JSON is a nice trick for getting structured output for your `TinyAgent` to parse.

# Tool Selection


Having your `TinyAgent` select a tool to use requires telling your Agent that it has access to the tools and how to use them. As such, we have to add a `prompt` property that generates that prompt for us with additional instructions.


In [10]:
from typing import Callable


class Tools:
    """Tool registry for the Agent."""

    def __init__(self):
        self.tools = {}

    def add_tool(self, name: str, func: Callable, description: str):
        """Register a tool that the Agent can use.

        Arguments:
            name: The name of the tool.
            func: The function implementing the tool.
            description: A description of the tool.
        """
        self.tools[name] = {"function": func, "description": description}

    @property
    def descriptions(self):
        """Get descriptions of all registered tools."""
        return "\n".join(
            f"`{tool}`: {self.tools[tool]['description']}" for tool in self.tools
        )

    @property
    def prompt(self):
        return f"""
# Tools

If needed, you can only use the following tools to assist you in completing tasks:

{self.descriptions}

To use a tool, respond with JSON: {{"tool": "name", "args": [...]}}
"""

When we now create our `Tools`, we can use `prompt` to view the prompt that will be given to your `TinyAgent`:

In [11]:
tools = Tools()
tools.add_tool("calculator", calculator, "Adds two numbers: calculator(a, b)")
tools.add_tool("get_weather", get_weather, "Gets weather: get_weather(city)")
print(tools.prompt)


# Tools

If needed, you can only use the following tools to assist you in completing tasks:

`calculator`: Adds two numbers: calculator(a, b)
`get_weather`: Gets weather: get_weather(city)

To use a tool, respond with JSON: {"tool": "name", "args": [...]}



As covered in the book, responsing with JSON is a nice trick for getting structured output for your `TinyAgent` to parse.

# Tool Calling

In the previous step, the model will most likely do something like:

```python
response = '{"tool": "calculator", "args": [value_1, value_2]}'
```

We will have to parse this response into proper JSON and then call the tools ourselves. The `TinyAgent` therefore doesn't actually call the tool, it has to be converted first before it can actually do something.

Tool calling in `Tools` requires adding the following functions:

* `is_tool_call` -- Checks whether a string contains a tool call
* `parse_tool_call` -- Converts a string to JSON so that we have a structured tool call
* `run_tool` -- Run a tool call

We need to first check whether a string actually contains a tool call, we then parse the tool call, and finally run it.

In [12]:
import json
from typing import Callable


class Tools:
    """Tool registry for the Agent."""

    def __init__(self):
        self.tools = {}

    def add_tool(self, name: str, func: Callable, description: str):
        """Register a tool that the Agent can use.

        Arguments:
            name: The name of the tool.
            func: The function implementing the tool.
            description: A description of the tool.
        """
        self.tools[name] = {"function": func, "description": description}

    @property
    def descriptions(self):
        """Get descriptions of all registered tools."""
        return "\n".join(
            f"`{tool}`: {self.tools[tool]['description']}" for tool in self.tools
        )

    @property
    def prompt(self):
        return f"""
# Tools

If needed, you can only use the following tools to assist you in completing tasks:

{self.descriptions}

To use a tool, respond with JSON: {{"tool": "name", "args": [...]}}
"""

    def is_tool_call(self, text: str) -> bool:
        """Check whether there is a tool call in `text`."""
        return '"tool":' in text or '"tool:"' in text

    def parse_tool_call(self, text: str) -> dict:
        """Parse a JSON tool call from text."""
        start, end = text.find("{"), text.rfind("}") + 1
        tool_call = json.loads(text[start:end])
        return tool_call

    def run_tool(self, tool_call: dict) -> any:
        """Run a registered tool.

        Arguments:
            tool_call: A parsed tool call dict with "tool" and "args" keys.
        """
        name, args = tool_call["tool"], tool_call.get("args", [])

        # Handle registered tools
        if name in self.tools:
            tool_func = self.tools[name]["function"]
            return tool_func(*args)

        # Handle intermediate_answer (allows LLM to respond without a real tool)
        if name == "intermediate_answer":
            return args

        return f"Tool '{name}' not found."


In [1]:
from illustrated_agents.chapters.ch5 import add_tool_annotated; add_tool_annotated

This is a very simplified way of handling this and in practice you might want to use actual structured JSON instead. However, since you might be using an LLM that does not handle structured JSON properly, we do it through this (rather simplified) example.

In [2]:
from illustrated_agents.chapters.ch5 import parse_tool_annotated; parse_tool_annotated

If you expect a more complex JSON structure with nested dictionaries, this will not work and requires a more complex function that first cleans up the text (which is not needed when using structured JSON; see the Chapter in the book for more information). 

In [1]:
from illustrated_agents.chapters.ch5 import run_tool_annotated; run_tool_annotated

---

💡 **NOTE**: The `run_tool` function also has a two lines for handling intermediate answers. Some LLMs will always attempt to use a tool, despite giving explicit instructions. The `"intermediate_answer"` is used in Chapter 6 where the LLM will **always** call a tool and use the `"intermediate_answer"` "tool" as a way to give back an answer without using a tool. For instance, creating a joke about chickens does not require a tool, so it may use the `"intermediate_answer"` "tool" to give back the answer. More on that later!

---

# Tool Output Processing (updating `agent.py`)

Processing the output of a tool is handled by the LLM, so we can add that step to your `TinyAgent`. Note that the main differences are at:

* `# 0. Tell the LLM about available tools` -> A system prompt was added. Note that we added it as a "user" role and not "system" role since Gemma 3 was not trained with the system role.
* `# 4. Tool parsing and execution` -> A couple of lines of code to check if a tool was called. If True, then we parse and execute the tool. Since this is still a single-turn example, the output of the tool is returned to the user.

In [5]:
from illustrated_agents import Memory


class TinyAgent:
    """A minimal, modular, and educational agent framework."""

    def __init__(self, llm: LLM, memory: Memory, tools: Tools):
        self.llm = llm
        self.memory = memory
        self.tools = tools
        self.planner = None  # Chapter 6: Add Planning
        self.reflector = None  # Chapter 6: Add Reflection

        # 0. Tell the LLM about available tools
        system_prompt = "You are a helpful assistant.\n\n" + self.tools.prompt
        self.memory.add("user", system_prompt)

    def run(self, task: str) -> str:
        """Run the agent on a task."""
        return self._step(task)

    def _step(self, task: str) -> str:
        """Perform a single step."""
        # 1. Every step, we add the user task to memory
        self.memory.add("user", task)

        # 2. Then, we generate a response based on the conversation history
        response = self.llm.generate(self.memory.get_messages())

        # 3. Finally, we add the assistant's response to memory
        self.memory.add("assistant", response)

        # 4. Tool parsing and execution
        if self.tools.is_tool_call(response):
            tool_call = self.tools.parse_tool_call(response)
            observation = self.tools.run_tool(tool_call)
            return str(observation)

        return response

Here is a nicer overview of the changes that we made to `agent.py`:

In [2]:
from illustrated_agents.chapters.ch5 import tinyagents_diff; tinyagents_diff

The `TinyAgent` can now be initialized with both the `Memory` and `Tools` modules:

In [18]:
# Tools
tools = Tools()
tools.add_tool("calculator", calculator, "Adds two numbers: calculator(a, b)")
tools.add_tool("get_weather", get_weather, "Gets weather: get_weather(city)")

# Memory
memory = Memory()

# Initialize Agent
agent = TinyAgent(llm=llm, memory=memory, tools=tools)

Let's check if the `TinyAgent` uses a tool when confronted with a question that may require one.

In [19]:
agent.run("What is 5.1281 plus 7.323?")

'12.4511'

It sure did! At least, I hope it did if you also used Gemma 3: 12B. If you didn't, it might behave differently. Either way, the prerendered output shows that a tool was correctly used!

Let's explore the traces of the interaction in case something went wrong:

In [20]:
agent.memory.get_messages()

[{'role': 'user',
  'content': 'You are a helpful assistant.\n\n\n# Tools\n\nIf needed, you can only use the following tools to assist you in completing tasks:\n\n`calculator`: Adds two numbers: calculator(a, b)\n`get_weather`: Gets weather: get_weather(city)\n\nTo use a tool, respond with JSON: {"tool": "name", "args": [...]}\n'},
 {'role': 'user', 'content': 'What is 5.1281 plus 7.323?'},
 {'role': 'assistant',
  'content': '{"tool": "calculator", "args": [5.1281, 7.323]}\n'}]

We can also check if your `TinyAgent` will answer questions that do not require tool usage:

In [21]:
agent.run("Hi! Tell me something about flamingos in two sentences.")

'Flamingos are vibrant pink birds known for their distinctive long legs and necks. They get their color from pigments in the algae and crustaceans they eat.'

It does! 😁 LLMs these days (Jan. 2026) are great in deciding themselves which tool to use and when. However, that does not mean it is not fallible. Describing hundreds of tools will likely fill up the context window too much and make it difficult for the LLM to decide if to use a tool and which one. 